# RelBench 代码分析与时序子图采样实践

这个 notebook 对应 `task.md` 里的三件事：

1. 用一个真实数据集演示 RelBench 的数据下载与缓存。
2. 追踪“关系数据库 -> 实体图”的实际代码路径。
3. 验证时序子图采样与测试阶段的时间/标签屏蔽机制。

本 notebook 选用 `rel-f1` + `driver-position`，因为它体量较小、下载快，而且能完整覆盖：

- `get_dataset(...)` / `get_task(...)`
- `Dataset.get_db(...)` 的时间截断
- `BaseTask.get_table(...)` 的测试标签屏蔽
- `make_pkey_fkey_graph(...)` 的异构图构造
- `NeighborLoader(..., time_attr=..., input_time=...)` 的时序采样

说明：当前服务器已经把 `rel-f1` 缓存到了 `~/.cache/relbench/rel-f1`。如果你换到新机器，第一次运行时把 `DOWNLOAD = True` 即可。

In [1]:
from __future__ import annotations

import inspect
from pathlib import Path
from pprint import pprint

import pandas as pd
from torch_frame import stype
from torch_geometric.loader import NeighborLoader

import relbench
from relbench.datasets import get_dataset
from relbench.tasks import get_task
from relbench.modeling.utils import get_stype_proposal
from relbench.modeling.graph import make_pkey_fkey_graph, get_node_train_table_input
from relbench.modeling.loader import TimestampSampler, LinkNeighborLoader

DOWNLOAD = False
DATASET_NAME = "rel-f1"
TASK_NAME = "driver-position"


REPO_ROOT = Path(relbench.__file__).resolve().parent.parent


def show_source(path: str, start: int, end: int) -> str:
    lines = (REPO_ROOT / path).read_text(encoding='utf-8').splitlines()
    selected = lines[start - 1 : end]
    return "\n".join(f"{i + start:4d}: {line}" for i, line in enumerate(selected))


def downgrade_text_stypes(col_to_stype_dict):
    """把 text_embedded 暂时降成 categorical，避免演示时额外下载文本编码模型。"""
    converted = {}
    replacements = []
    for table, mapping in col_to_stype_dict.items():
        new_mapping = dict(mapping)
        for col, s in list(new_mapping.items()):
            if s == stype.text_embedded:
                new_mapping[col] = stype.categorical
                replacements.append((table, col))
        converted[table] = new_mapping
    return converted, replacements

/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. 下载数据集与任务

对应源码入口：

- `relbench/datasets/__init__.py:68-109` 负责数据集下载与 `get_dataset(...)`
- `relbench/tasks/__init__.py:72-113` 负责任务下载与 `get_task(...)`

这里先把数据集和任务对象加载出来，再确认缓存路径、验证/测试分割时间。

In [2]:
dataset = get_dataset(DATASET_NAME, download=DOWNLOAD)
task = get_task(DATASET_NAME, TASK_NAME, download=DOWNLOAD)

print('dataset cache_dir:', dataset.cache_dir)
print('task cache_dir   :', task.cache_dir)
print('val timestamp    :', dataset.val_timestamp)
print('test timestamp   :', dataset.test_timestamp)
print('task class       :', task.__class__.__name__)
print('entity table/col :', task.entity_table, task.entity_col)
print('target/time col  :', task.target_col, task.time_col)
print('timedelta        :', task.timedelta)
print('metrics          :', [metric.__name__ for metric in task.metrics])

dataset cache_dir: /home/xuewenqi/.cache/relbench/rel-f1
task cache_dir   : /home/xuewenqi/.cache/relbench/rel-f1/tasks/driver-position
val timestamp    : 2005-01-01 00:00:00
test timestamp   : 2010-01-01 00:00:00
task class       : DriverPositionTask
entity table/col : drivers driverId
target/time col  : position date
timedelta        : 60 days 00:00:00
metrics          : ['r2', 'mae', 'rmse']


## 2. 时间截断：数据库层怎么防止未来信息泄漏

核心代码：

- `relbench/base/dataset.py:81-131`
- `relbench/base/database.py:66-73`
- `relbench/base/table.py:107-121`

默认的 `dataset.get_db()` 会调用 `db.upto(self.test_timestamp)`，把所有带时间列的表裁剪到 `test_timestamp` 为止。

下面直接比较：

- 默认数据库 `dataset.get_db()`
- 完整数据库 `dataset.get_db(upto_test_timestamp=False)`

In [3]:
db = dataset.get_db()
full_db = dataset.get_db(upto_test_timestamp=False)

summary_rows = []
for table_name in db.table_dict:
    cutoff_table = db.table_dict[table_name]
    full_table = full_db.table_dict[table_name]
    summary_rows.append({
        'table': table_name,
        'rows_cutoff': len(cutoff_table.df),
        'rows_full': len(full_table.df),
        'time_col': cutoff_table.time_col,
        'future_rows_removed': len(full_table.df) - len(cutoff_table.df),
    })

pd.DataFrame(summary_rows).sort_values('future_rows_removed', ascending=False)

Loading Database object from /root/.cache/relbench/rel-f1/db...
Done in 0.02 seconds.
Loading Database object from /root/.cache/relbench/rel-f1/db...


Done in 0.23 seconds.


,table,rows_cutoff,rows_full,time_col,future_rows_removed
7,standings,28115,34124,date,6009
4,results,20323,26080,date,5757
3,qualifying,4082,9815,date,5733
6,constructor_results,9408,12290,date,2882
0,constructor_standings,10170,13051,date,2881
2,races,820,1101,date,281
1,circuits,77,77,None,0
5,constructors,211,211,None,0
8,drivers,857,857,None,0


可以看到，带时间列的表在默认数据库里会丢掉 `test_timestamp` 之后的行；这就是 RelBench 第一层“时间屏蔽”。

再看源码片段：

In [4]:
print(show_source('relbench/base/dataset.py', 81, 131))
print() 
print(show_source('relbench/base/database.py', 66, 73))
print()
print(show_source('relbench/base/table.py', 107, 121))

  81:     def get_db(self, upto_test_timestamp=True) -> Database:
  82:         r"""Return the database object.
  83: 
  84:         The returned database object is cached in memory.
  85: 
  86:         Args:
  87:             upto_test_timestamp: If True, only return rows upto test_timestamp.
  88: 
  89:         Returns:
  90:             Database: The database object.
  91: 
  92:         `upto_test_timestamp` is True by default to prevent test leakage.
  93:         """
  94: 
  95:         db_path = f"{self.cache_dir}/db"
  96:         if self.cache_dir and Path(db_path).exists() and any(Path(db_path).iterdir()):
  97:             print(f"Loading Database object from {db_path}...")
  98:             tic = time.time()
  99:             db = Database.load(db_path)
 100:             toc = time.time()
 101:             print(f"Done in {toc - tic:.2f} seconds.")
 102: 
 103:         else:
 104:             print("Making Database object from scratch...")
 105:             print(
 106: 

## 3. 任务表怎么构造，测试标签又是怎么被屏蔽的

RelBench 第二层“屏蔽”发生在任务表：

- `BaseTask.get_table(...)` 默认会对 `test` split 执行 `_mask_input_cols(...)`
- 这样测试集只保留输入列，不把标签列暴露给模型

对 `driver-position` 而言，任务定义在 `relbench/tasks/f1.py:19-67`。它会根据时间窗口，聚合未来 60 天里每个车手的平均完赛位置。

In [5]:
train_table = task.get_table('train')
val_table = task.get_table('val')
test_table_masked = task.get_table('test')
test_table_full = task.get_table('test', mask_input_cols=False)

print('train rows:', len(train_table.df))
print('val rows  :', len(val_table.df))
print('test rows :', len(test_table_masked.df))
print()
print('masked test columns:', test_table_masked.df.columns.tolist())
print('full test columns  :', test_table_full.df.columns.tolist())
print()
print(test_table_full.df.head())

train rows: 7453
val rows  : 499
test rows : 760

masked test columns: ['date', 'driverId']
full test columns  : ['date', 'driverId', 'position']

        date  driverId   position
0 2016-05-29       835  17.000000
1 2016-03-30         3  12.333333
2 2016-03-30       807  17.750000
3 2016-03-30       831  11.250000
4 2016-01-30       830  15.000000


In [6]:
print(show_source('relbench/base/task_base.py', 157, 209))
print()
print(show_source('relbench/tasks/f1.py', 19, 67))

 157:     @lru_cache(maxsize=None)
 158:     def get_table(self, split, mask_input_cols=None):
 159:         r"""Get a table for a split.
 160: 
 161:         Args:
 162:             split: The split to get the table for. One of "train", "val", or "test".
 163:             mask_input_cols: If True, keep only the input columns in the table. If
 164:                 None, mask the input columns only for the test split. This helps
 165:                 prevent data leakage.
 166: 
 167:         Returns:
 168:             The task table for the split.
 169: 
 170:         The table is cached in memory.
 171:         """
 172: 
 173:         if mask_input_cols is None:
 174:             mask_input_cols = split == "test"
 175: 
 176:         table_path = f"{self.cache_dir}/{split}.parquet"
 177:         if self.cache_dir and Path(table_path).exists():
 178:             table = Table.load(table_path)
 179:         else:
 180:             print(f"Making task table for {split} split from scratc

结论：

- `DriverPositionTask.make_table(...)` 先用时间窗口生成监督信号。
- `task.get_table('test')` 再把标签列 `position` 藏起来。
- 如果你需要评估或调试，才显式调用 `mask_input_cols=False`。

## 4. 关系数据库如何变成实体图

关键函数是 `relbench/modeling/graph.py:20-111` 的 `make_pkey_fkey_graph(...)`。

它做了 4 件事：

1. 遍历数据库里的每张表。
2. 检查主键列是否已经被重排成连续整数。
3. 把主键/外键从输入特征里移除，只保留真正的属性列做 `TensorFrame`。
4. 对每个外键列建立两种边：
   - 正向边：`fkey -> pkey`
   - 反向边：`pkey -> fkey`（边名带 `rev_`，方便 PyG 识别）

为了让 notebook 运行得更轻量，这里把自动推断成 `text_embedded` 的列临时降成 `categorical`，避免额外下载文本 embedding 模型。结构分析本身不受影响。

In [7]:
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict, replacements = downgrade_text_stypes(col_to_stype_dict)
print('downgraded text columns -> categorical:')
pprint(replacements)

data, col_stats_dict = make_pkey_fkey_graph(db, col_to_stype_dict)

print('node types:')
pprint(data.node_types)
print() 
print('number of edge types:', len(data.edge_types))
print('first 8 edge types:')
pprint(data.edge_types[:8])
print()
print('drivers TensorFrame rows:', data['drivers'].tf.num_rows)
print('results has time attr:', hasattr(data['results'], 'time'))
print('results time length  :', data['results'].time.numel())

downgraded text columns -> categorical:
[('circuits', 'circuitRef'),
 ('circuits', 'name'),
 ('circuits', 'location'),
 ('circuits', 'country'),
 ('races', 'name'),
 ('constructors', 'constructorRef'),
 ('constructors', 'name'),
 ('constructors', 'nationality'),
 ('drivers', 'driverRef'),
 ('drivers', 'code'),
 ('drivers', 'forename'),
 ('drivers', 'surname'),
 ('drivers', 'nationality')]


/root/miniconda3/lib/python3.12/site-packages/torch_frame/data/stats.py:177: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ser = pd.to_datetime(ser, format=time_format)
/root/miniconda3/lib/python3.12/site-packages/torch_frame/data/mapper.py:291: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ser = pd.to_datetime(ser, format=self.format, errors='coerce')


node types:
['constructor_standings',
 'circuits',
 'races',
 'qualifying',
 'results',
 'constructors',
 'constructor_results',
 'standings',
 'drivers']

number of edge types: 26
first 8 edge types:
[('constructor_standings', 'f2p_raceId', 'races'),
 ('races', 'rev_f2p_raceId', 'constructor_standings'),
 ('constructor_standings', 'f2p_constructorId', 'constructors'),
 ('constructors', 'rev_f2p_constructorId', 'constructor_standings'),
 ('races', 'f2p_circuitId', 'circuits'),
 ('circuits', 'rev_f2p_circuitId', 'races'),
 ('qualifying', 'f2p_raceId', 'races'),
 ('races', 'rev_f2p_raceId', 'qualifying')]

drivers TensorFrame rows: 857
results has time attr: True
results time length  : 20323


In [8]:
print(show_source('relbench/modeling/graph.py', 20, 111))

  20: def make_pkey_fkey_graph(
  21:     db: Database,
  22:     col_to_stype_dict: Dict[str, Dict[str, stype]],
  23:     text_embedder_cfg: Optional[TextEmbedderConfig] = None,
  24:     cache_dir: Optional[str] = None,
  25: ) -> Tuple[HeteroData, Dict[str, Dict[str, Dict[StatType, Any]]]]:
  26:     r"""Given a :class:`Database` object, construct a heterogeneous graph with primary-
  27:     foreign key relationships, together with the column stats of each table.
  28: 
  29:     Args:
  30:         db: A database object containing a set of tables.
  31:         col_to_stype_dict: Column to stype for
  32:             each table.
  33:         text_embedder_cfg: Text embedder config.
  34:         cache_dir: A directory for storing materialized tensor
  35:             frames. If specified, we will either cache the file or use the
  36:             cached file. If not specified, we will not use cached file and
  37:             re-process everything from scratch without saving the

你可以把 `HeteroData` 理解成：

- 每张表是一类节点（node type）
- 每个外键列是一类关系（edge type）
- 特征来自去掉主键/外键后的表属性
- 时间列会额外挂到 `data[table_name].time`

这也是“把数据库变成实体图”的最核心实现。

## 5. 子图采样：NeighborLoader 实际抽出了什么

对实体任务，RelBench 走的是 `get_node_train_table_input(...)` + `torch_geometric.loader.NeighborLoader(...)` 这条路径：

- `get_node_train_table_input(...)` 把任务表转成 `(input_nodes, input_time, y)`
- `NeighborLoader(..., time_attr='time', input_time=...)` 再围绕这些种子节点做时序邻居采样

下面抽一个最小 batch，直接看采样结果。

In [9]:
table_input = get_node_train_table_input(train_table, task)
loader = NeighborLoader(
    data,
    num_neighbors=[8, 4],
    time_attr='time',
    input_nodes=table_input.nodes,
    input_time=table_input.time,
    transform=table_input.transform,
    batch_size=4,
    shuffle=False,
)

batch = next(iter(loader))
seed_times = table_input.time[:4]

print('seed input ids:', batch[task.entity_table].input_id.tolist())
print('seed labels   :', batch[task.entity_table].y.tolist())
print('seed times    :', seed_times.tolist())

sampling_rows = []
for node_type in batch.node_types:
    info = batch[node_type]
    row = {
        'node_type': node_type,
        'num_nodes': int(info.num_nodes),
        'has_time': 'time' in info,
    }
    if 'time' in info and info['time'].numel() > 0:
        row['time_min'] = int(info['time'].min())
        row['time_max'] = int(info['time'].max())
        row['seed_time_max'] = int(seed_times.max())
        row['all_le_seed_max'] = bool((info['time'] <= seed_times.max()).all())
    sampling_rows.append(row)

pd.DataFrame(sampling_rows)

seed input ids: [0, 1, 2, 3]
seed labels   : [10.75, 12.0, 15.0, 9.0]
seed times    : [1088985600, 1088985600, 1078617600, 1073433600]


,node_type,num_nodes,has_time,time_min,time_max,seed_time_max,all_le_seed_max
0,constructor_standings,0,True,NaN,NaN,NaN,NaN
1,circuits,0,False,NaN,NaN,NaN,NaN
2,races,69,True,920764800.0,1.088899e+09,1.088986e+09,True
3,qualifying,27,True,920678400.0,1.088813e+09,1.088986e+09,True
4,results,32,True,932860800.0,1.088899e+09,1.088986e+09,True
5,constructors,8,False,NaN,NaN,NaN,NaN
6,constructor_results,0,True,NaN,NaN,NaN,NaN
7,standings,32,True,931651200.0,1.088899e+09,1.088986e+09,True
8,drivers,4,False,NaN,NaN,NaN,NaN


这张表里最重要的是 `all_le_seed_max`：

- 对所有真正带时间戳的节点类型，它都应该是 `True`
- 这说明采样出来的邻居不会晚于当前 seed 对应的查询时间

这就是时序子图采样在运行时的实际效果。

## 6. 时间屏蔽采样在代码里的完整链路

实体任务用的是 PyG 的 `NeighborLoader`。

推荐/链接预测任务则走 `relbench/modeling/loader.py` 里的自定义 loader：

- `TimestampSampler`：如果 `share_same_time=True`，把同一时间戳的样本编成一个 batch
- `LinkNeighborLoader`：
  - 从每个源点抽 1 个正样本目的点
  - 再随机抽负样本目的点
  - 分别对 `src / pos_dst / neg_dst` 做子图采样

下面直接查看源码片段。

In [10]:
print(show_source('relbench/modeling/loader.py', 87, 123))
print()
print(show_source('relbench/modeling/loader.py', 166, 289))

  87: class TimestampSampler(Sampler[int]):
  88:     r"""A TimestampSampler that samples rows from the same timestamp."""
  89: 
  90:     def __init__(
  91:         self,
  92:         timestamp: Tensor,
  93:         batch_size: int,
  94:     ):
  95:         super().__init__()
  96:         self.batch_size = batch_size
  97:         self.time_dict = {
  98:             int(time): (timestamp == time).nonzero().view(-1)
  99:             for time in timestamp.unique()
 100:         }
 101:         self.num_batches = sum(
 102:             [indices.numel() // batch_size for indices in self.time_dict.values()]
 103:         )
 104: 
 105:     def __iter__(self) -> Iterator[List[int]]:
 106:         all_batches = []
 107:         for indices in self.time_dict.values():
 108:             # Random shuffle values:
 109:             indices = indices[torch.randperm(indices.numel())]
 110:             batches = torch.split(indices, self.batch_size)
 111:             for batch in batches:
 

补充一个非常容易忽略的点：如果你想把“历史标签表”也并进图里做自回归特征，RelBench 不是直接把标签塞进去，而是先做时间审查。

示例代码在 `examples/gnn_entity.py:100-122`：

- 把 train/val 标签表拼起来
- 再把标签时间 `label_time += task.timedelta`
- 只有到标签真正“应当可见”的时间点，模型才能采样到它

这属于另一种更严格的时间屏蔽（time censoring）。

In [11]:
print(show_source('examples/gnn_entity.py', 100, 122))

 100: db = dataset.get_db()
 101: # add (time-censored) labels tables to the db
 102: for task_name in tasks_to_add:
 103:     t = get_task(args.dataset, task_name, download=bool(args.download))
 104:     if not isinstance(t, EntityTask):
 105:         continue
 106:     labels_table_name = f"{task_name}_labels"
 107:     label_df = pd.concat(
 108:         [
 109:             t.get_table("train").df,
 110:             t.get_table("val").df,
 111:             # test set not included b/c labels are not revealed
 112:         ]
 113:     )
 114:     # time-censoring labels: we add timedelta to the time column to ensure that
 115:     # the labels become available at the appropriate time (i.e. no leakage)
 116:     label_df[t.time_col] = label_df[t.time_col] + t.timedelta
 117:     db.table_dict[labels_table_name] = Table(
 118:         df=label_df,
 119:         fkey_col_to_pkey_table={t.entity_col: t.entity_table},
 120:         pkey_col=None,
 121:         time_col=t.time_col,
 122:   

## 7. 最后总结：你应该如何在代码里定位这三部分

### A. 数据库变实体图

从这里开始读：

- `relbench/modeling/graph.py:20-111`
- `relbench/modeling/utils.py:30-62`
- `relbench/base/database.py:84-122`

重点看：

- 主键/外键什么时候被重排成连续索引
- 哪些列被保留成输入特征
- 一个外键如何生成正反两种边

### B. 子图采样

实体任务：

- `relbench/modeling/graph.py:147-178`
- `torch_geometric.loader.NeighborLoader`

链接任务：

- `relbench/modeling/graph.py:199-228`
- `relbench/modeling/loader.py:166-289`

重点看：

- seed 节点和 seed 时间从任务表怎么来
- 每层 hop 采多少邻居
- 正样本/负样本怎么组织成 batch

### C. 时间屏蔽 / 时间约束采样

数据截断：

- `relbench/base/dataset.py:81-131`
- `relbench/base/database.py:66-73`
- `relbench/base/table.py:107-121`

测试标签屏蔽：

- `relbench/base/task_base.py:157-209`

时序邻居采样：

- `NeighborLoader(..., time_attr='time', input_time=...)`
- `relbench/modeling/loader.py:87-123`
- `examples/gnn_entity.py:114-116`

如果你下一步想自己改模型，建议直接从 `examples/gnn_entity.py` 或 `examples/idgnn_recommendation.py` 往下跟，因为这两个脚本已经把 RelBench 的数据侧和 PyG 的采样侧接起来了。